# Patent DPR Colab Pro Training

Run this notebook on a Colab GPU runtime or a VSCode Colab-connected kernel.

Local Codex prepares `HYU-DPR_colab_input.tar.gz`; this notebook mounts Drive, extracts the clean bundle, runs smoke/profile checks, then starts the 40-epoch DPR training run.

## Runtime Contract

- Drive path: `/content/drive/MyDrive/HYU-DPR/`
- Bundle: `HYU-DPR_colab_input.tar.gz`
- Smoke output: `outputs/patent_dpr_colab_smoke/`
- Full output: `outputs/patent_dpr_colab_pro/`
- A100/L4 target: actual `batch_size=128`, `gradient_accumulation_steps=1`, `epoch=40`
- T4 fallback: `batch_size=64, grad_accum=2` or `batch_size=32, grad_accum=4`

This DPR clone uses NVIDIA Apex for `fp16=True`, so the default cells run fp32. If `batch_size=128` fails only due to memory, install Apex and rerun with `--fp16`.

In [1]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/HYU-DPR')
BUNDLE = DRIVE_ROOT / 'HYU-DPR_colab_input.tar.gz'
WORKSPACE = Path('/content/HYU-DPR')

print('DRIVE_ROOT =', DRIVE_ROOT)
print('BUNDLE =', BUNDLE)
print('WORKSPACE =', WORKSPACE)

DRIVE_ROOT = /content/drive/MyDrive/HYU-DPR
BUNDLE = /content/drive/MyDrive/HYU-DPR/HYU-DPR_colab_input.tar.gz
WORKSPACE = /content/HYU-DPR


In [2]:
from google.colab import drive
drive.mount('/content/drive')

# The notebook expects /content/drive/MyDrive/HYU-DPR/HYU-DPR_colab_input.tar.gz.
# If the file was uploaded to a different Drive folder, search MyDrive and reuse that path.
if not BUNDLE.exists():
    candidates = list(Path('/content/drive/MyDrive').rglob('HYU-DPR_colab_input.tar.gz'))
    if candidates:
        BUNDLE = candidates[0]
        DRIVE_ROOT = BUNDLE.parent
        print('Found bundle at:', BUNDLE)
        print('Using DRIVE_ROOT:', DRIVE_ROOT)
    else:
        print('Expected path:', BUNDLE)
        print('No HYU-DPR_colab_input.tar.gz found under /content/drive/MyDrive')
        print('Top-level MyDrive files/folders:')
        for p in sorted(Path('/content/drive/MyDrive').iterdir())[:50]:
            print(' -', p)
        raise FileNotFoundError('Upload HYU-DPR_colab_input.tar.gz into the mounted Google account, preferably MyDrive/HYU-DPR/.')

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Bundle size MB:', round(BUNDLE.stat().st_size / 1024 / 1024, 2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found bundle at: /content/drive/MyDrive/HYU-DPR/colab/HYU-DPR_colab_input.tar.gz
Using DRIVE_ROOT: /content/drive/MyDrive/HYU-DPR/colab
Bundle size MB: 14.87


In [3]:
import shutil
import tarfile

if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir(parents=True, exist_ok=True)

with tarfile.open(BUNDLE, 'r:gz') as tar:
    tar.extractall(WORKSPACE)

print('Extracted to:', WORKSPACE)
print('DPR train script:', (WORKSPACE / 'DPR-main' / 'train_dense_encoder.py').exists())
print('Patent train data:', (WORKSPACE / 'data' / 'patent' / 'train.json').exists())

/tmp/ipykernel_27428/23059713.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(WORKSPACE)


Extracted to: /content/HYU-DPR
DPR train script: True
Patent train data: True


## Install DPR Dependencies

In [4]:
%cd /content/HYU-DPR/DPR-main
!python -m pip install --upgrade pip setuptools wheel
!python -m pip install "numpy<2" "transformers>=4.3" "hydra-core>=1.0.0" "omegaconf>=2.0.1" faiss-cpu jsonlines soundfile editdistance wget "spacy>=2.1.8"
!python -m pip install -e .

/content/HYU-DPR/DPR-main
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.2 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 103.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 13

In [6]:
import os
import subprocess
import sys

os.chdir("/content/HYU-DPR")
sys.path.insert(0, "/content/HYU-DPR/DPR-main")

import torch
import transformers
import dpr

print("cwd", os.getcwd())
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("dpr import ok")
subprocess.run(["nvidia-smi"], check=False)
subprocess.run([sys.executable, "colab/colab_runner.py", "gpu"], check=True)

cwd /content/HYU-DPR
torch 2.10.0+cu128
cuda True
transformers 5.0.0
dpr import ok


CompletedProcess(args=['/usr/bin/python3', 'colab/colab_runner.py', 'gpu'], returncode=0)

## Smoke Test

This is a tiny end-to-end run. It verifies data loading, DPR import, training, checkpoint writing, and `run_metadata.json` creation.

In [7]:
%cd /content/HYU-DPR
!python colab/colab_runner.py smoke

/content/HYU-DPR
$ /usr/bin/python3 train_dense_encoder.py 'train_datasets=[patent_train]' 'dev_datasets=[patent_dev]' datasets.patent_train.file=/content/HYU-DPR/data/patent_colab_smoke/train.json datasets.patent_dev.file=/content/HYU-DPR/data/patent_colab_smoke/dev.json train=biencoder_patent_colab_smoke encoder=hf_mbert do_lower_case=False output_dir=/content/drive/MyDrive/HYU-DPR/outputs/patent_dpr_colab_smoke fp16=False hydra.run.dir=. val_av_rank_start_epoch=100
/content/HYU-DPR/DPR-main/train_dense_encoder.py:747: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path="conf", config_name="biencoder_train_cfg")
[140085600969344] 2026-05-17 11:34:58,730 [INFO] root: Sys.argv: ['train_dense_encoder.py', 'train_datasets=[patent_train]', 'dev_datasets=[patent_dev]', 'datasets.patent_train.file=/content/HYU-DPR/data/patent_colab_smoke/train.json', 'datasets.patent_

## Batch Feasibility Check

Run this before the full 40-epoch job. `auto` chooses `128` for A100/L4 and `64` otherwise.

In [8]:
%cd /content/HYU-DPR
!python colab/colab_runner.py profile-check --profile auto --steps 20

/content/HYU-DPR
$ /usr/bin/python3 train_dense_encoder.py 'train_datasets=[patent_train]' 'dev_datasets=[patent_dev]' datasets.patent_train.file=/content/HYU-DPR/data/patent_colab_profile_128/train.json datasets.patent_dev.file=/content/HYU-DPR/data/patent_colab_profile_128/dev.json train=biencoder_patent_colab_128 encoder=hf_mbert do_lower_case=False output_dir=/content/drive/MyDrive/HYU-DPR/outputs/patent_dpr_colab_profile_128 fp16=False hydra.run.dir=. train.num_train_epochs=1 train.warmup_steps=1 val_av_rank_start_epoch=100
/content/HYU-DPR/DPR-main/train_dense_encoder.py:747: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path="conf", config_name="biencoder_train_cfg")
[139004363207296] 2026-05-17 11:35:56,189 [INFO] root: Sys.argv: ['train_dense_encoder.py', 'train_datasets=[patent_train]', 'dev_datasets=[patent_dev]', 'datasets.patent_train.file=/content/

If the profile check fails due to OOM, run fallback checks.

In [9]:
%cd /content/HYU-DPR
# !python colab/colab_runner.py profile-check --profile 64 --steps 20
# !python colab/colab_runner.py profile-check --profile 32 --steps 20

/content/HYU-DPR


## Full Training

For A100/L4 paper-style reproduction, run the `--profile 128` cell. For Colab Pro auto selection with fallback, run the auto-fallback cell instead.

In [4]:
!nvidia-smi

Sun May 17 11:50:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   64C    P0            374W /  400W |   66064MiB /  81920MiB |    100%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [5]:
!kill -9 31530
!nvidia-smi

Sun May 17 11:50:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   58C    P0             89W /  400W |   66064MiB /  81920MiB |     99%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [6]:
%cd /content/HYU-DPR
!python colab/colab_runner.py train --profile 128

/content/HYU-DPR
$ /usr/bin/python3 train_dense_encoder.py 'train_datasets=[patent_train]' 'dev_datasets=[patent_dev]' datasets.patent_train.file=/content/HYU-DPR/data/patent/train.json datasets.patent_dev.file=/content/HYU-DPR/data/patent/dev.json train=biencoder_patent_colab_128 encoder=hf_mbert do_lower_case=False output_dir=/content/drive/MyDrive/HYU-DPR/outputs/patent_dpr_colab_pro fp16=False hydra.run.dir=.
/content/HYU-DPR/DPR-main/train_dense_encoder.py:747: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path="conf", config_name="biencoder_train_cfg")
[140028479627904] 2026-05-17 11:50:55,293 [INFO] root: Sys.argv: ['train_dense_encoder.py', 'train_datasets=[patent_train]', 'dev_datasets=[patent_dev]', 'datasets.patent_train.file=/content/HYU-DPR/data/patent/train.json', 'datasets.patent_dev.file=/content/HYU-DPR/data/patent/dev.json', 'train=biencoder_pa

In [7]:
%cd /content/HYU-DPR
# Use this only when the GPU is not A100/L4, or when profile 128 fails.
# !python colab/colab_runner.py train --profile auto --auto-fallback

/content/HYU-DPR


## Required Artifacts

After full training, these files must exist in Drive:

- `/content/drive/MyDrive/HYU-DPR/outputs/patent_dpr_colab_pro/dpr_biencoder.0`
- `/content/drive/MyDrive/HYU-DPR/outputs/patent_dpr_colab_pro/train_dense_encoder.log`
- `/content/drive/MyDrive/HYU-DPR/outputs/patent_dpr_colab_pro/run_metadata.json`

Download/copy that folder back to the local repo as `outputs/patent_dpr_colab_pro/`, then run `python3 tools/verify_colab_artifacts.py --run-dir outputs/patent_dpr_colab_pro` locally.

In [8]:
from pathlib import Path
out = Path('/content/drive/MyDrive/HYU-DPR/outputs/patent_dpr_colab_pro')
for name in ['dpr_biencoder.0', 'train_dense_encoder.log', 'run_metadata.json']:
    path = out / name
    print(name, path.exists(), round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else None)

dpr_biencoder.0 True 4062.24
train_dense_encoder.log True 0.63
run_metadata.json True 0.0
